# Notebook consumers: DuckDB analytics + the recommendation engine

Both stores stay useful from a notebook, per PLAN.md:

- **DuckDB / Parquet** (opened *read-only*, so it coexists with the API and ingest) for ad-hoc analytics — this is what DuckDB is best at.
- **SQLite** through `recommend.py` for the four recommendation query types, same code path as the CLI and FastAPI app.

Run from the repo root (`uv run jupyter ...` or point your kernel at `.venv`) after an ingest + `recindex.py`.

In [ ]:
import sys
from pathlib import Path

repo = Path.cwd().resolve()
if repo.name == "examples":
    repo = repo.parent
sys.path.insert(0, str(repo))
out = repo / "data" / "processed"

## Analytics side (DuckDB, read-only)

In [ ]:
import duckdb

ddb = duckdb.connect(str(out / "jobs.duckdb"), read_only=True)
ddb.sql("""
    SELECT job_category, COUNT(*) AS jobs,
           round(avg(yearly_max_compensation)) AS avg_max_comp
    FROM jobs WHERE NOT is_expired
    GROUP BY 1 ORDER BY jobs DESC LIMIT 15
""")

In [ ]:
# Same queries work straight off the Parquet export (no DB file needed):
ddb.sql(f"""
    SELECT seniority_level,
           quantile_cont(yearly_min_compensation, 0.5) AS p50_min,
           quantile_cont(yearly_max_compensation, 0.5) AS p50_max
    FROM read_parquet('{(out / "jobs.parquet").as_posix()}')
    WHERE yearly_min_compensation IS NOT NULL
    GROUP BY 1 ORDER BY p50_max DESC
""") if (
    out / "jobs.parquet"
).exists() else "no jobs.parquet (run ingest with --parquet)"

## Recommendation side (SQLite, same functions as CLI/API)

In [ ]:
import recommend
from recommend import Filters

con = recommend.get_connection(out / "jobs.sqlite")
recommend.search_keywords(
    con,
    "staff platform engineer kubernetes",
    Filters(workplace_types=["Remote"], min_comp=150_000),
    k=5,
)

In [ ]:
resume_md = """
# Jane Doe\n\nBackend engineer, 6 years: Python, PostgreSQL, Kubernetes,
AWS. Built APIs and data pipelines.
"""
results, keywords = recommend.match_resume(con, resume_md, k=5)
print("keywords:", keywords)
[(r["title"], r["company_name"], round(r["score"], 4)) for r in results]

In [ ]:
# "More like this" for the first hit:
recommend.similar_jobs(con, results[0]["requisition_id"], k=5) if results else []